In [ ]:
import os
import git
import pandas as pd
from fake_useragent import UserAgent
import requests
import datetime
import json
from utils.file_utils import dir_check, safe_open
from utils.parser_utils import get_query, get_syntax_tree
from tree_sitter import Tree
from utils.sample_utils import (
    find_parent_fn_node,
    node_of_unsafe_func,
    node_of_pub_func,
)
crates_info_file = "data/crate_meta/crates_info.csv"

## Crawl crates info from crates.io

In [ ]:
# get the crates infromation from crates.io
def craw_crates_list(crates_info_file):
    df = pd.DataFrame(
        columns=[
            "name",
            "newest_version",
            "recent_downloads",
            "created_at",
            "updated_at",
            "repository",
        ]
    )
    headers = {
        "User-Agent": UserAgent().random,
        "referer": "https://crates.io/crates?sort=downloads",
    }
    total = 0
    n = 0
    error_num = 0
    while 1:
        url = (
            "https://crates.io/api/v1/crates?page="
            + str(n + 1)
            + "&per_page=100&sort=downloads"
        )
        try:
            page_text = requests.get(url=url, headers=headers).text
            crates = json.loads(page_text)["crates"]
        except:
            print("Error in {}, retry-{} ".format(n + 1, error_num))
            error_num += 1
            if error_num < 10:
                continue
            else:
                break
        data_lists = [[] for i in range(6)]
        for crate in crates:
            data_lists[0].append(crate["name"])
            data_lists[1].append(crate["newest_version"])
            data_lists[2].append(crate["recent_downloads"])
            data_lists[3].append(crate["created_at"])
            data_lists[4].append(crate["updated_at"])
            data_lists[5].append(crate["repository"])
        dic = {
            "name": data_lists[0],
            "newest_version": data_lists[1],
            "recent_downloads": data_lists[2],
            "created_at": data_lists[3],
            "updated_at": data_lists[4],
            "repository": data_lists[5],
        }
        df2 = pd.DataFrame(dic)
        df = pd.concat([df, df2], axis=0)
        print("Crawl page {}, get {} info".format(n + 1, len(df2)))
        error_num = 0
        total += len(df2)
        n += 1
        if len(df2) < 50:
            print("Get {} crates in total".format(total))
            break

    # crates are sorted with "recent_downloads" descendingly
    df.to_csv(crates_info_file, index=None)


craw_crates_list(crates_info_file)

## Helper functions

In [2]:
# filter the crates (top-n)
def filter_crates(crates_info_file, result_file, target_range=(0,500)):
    data = pd.read_csv(crates_info_file, sep=",", header=0, usecols=[0, 2, 4, 5])
    array = data.values[0::, 0::]
    repo_list = []
    error_count = [0, 0, 0, 0]
    # Rust 2018: 2018-12-07
    RUST_2018 = datetime.datetime(2018, 12, 7, 0, 0, 0)
    for i in range(target_range[0], target_range[1]):
        update = datetime.datetime.strptime(str(array[i][2])[:19], "%Y-%m-%dT%H:%M:%S")
        if update <= RUST_2018:
            error_count[0] += 1
            continue

        if len(str(array[i][3])) < 5:
            error_count[1] += 1
            continue
        url = str(array[i][3]).split("/")
        if "github.com" in url[2]:
            repo = url[3] + "/" + url[4].replace(".git", "")
            if repo not in repo_list:
                repo_list.append(repo)
            else:
                error_count[3] += 1
        else:
            error_count[2] += 1

    print(len(repo_list))
    print(error_count)
    open(result_file, "w").write("\n".join(repo_list))
    return repo_list



def clone_repo(repo_path, repo_dir):
    # clone repo from github
    repo_path = repo_path.strip()
    fail_count = 0
    while 1:
        try:
            repo_url = "https://github.com/" + repo_path + ".git"
            if os.path.exists(repo_dir):
                break
            git.Repo.clone_from(url=repo_url, to_path=repo_dir, depth=1)
            break
        except:
            if fail_count < 10:
                fail_count += 1
                continue
            else:
                print(repo_path + " clone failed!")
                break

In [3]:
unsafe_block_pattern = "(unsafe_block) @unsafe_block"
query = get_query(unsafe_block_pattern)

def find_risky_func(repository_dir: str):
    samples = list()
    for dir_path, _, files in os.walk(repository_dir):
        for file_name in files:
            if file_name[-3:] != ".rs":
                continue
            file_path = os.path.join(dir_path, file_name)
            relative_file = os.path.relpath(os.path.join(dir_path, file_name), repository_dir)
            if "test" in file_path or "example" in file_path or "bench" in file_path:
                continue
            tree: Tree = get_syntax_tree(file_path)
            captures = query.captures(tree.root_node)
            for node, _ in captures:
                # tranverse cursor to father function
                fn_node = find_parent_fn_node(node)
                if fn_node is None:
                    continue

                # filter unsafe block in unsafe function
                if node_of_unsafe_func(fn_node):
                    continue

                # filter private functions
                if not node_of_pub_func(fn_node):
                    continue

                duplicated = False
                for s in samples:
                    if s["relative_file"] == relative_file and s["start_line"] == fn_node.start_point[0]:
                        duplicated = True
                        break
                if duplicated:
                    continue

                samples.append(
                    {
                        "relative_file": relative_file,
                        "start_line": fn_node.start_point[0], 
                        "start_byte": fn_node.start_byte,
                        "end_line": fn_node.end_point[0],
                        "end_byte": fn_node.end_byte,
                    }
                )
    print(f"Got {len(samples)} risky function in {repository_dir}")
    return samples


def find_unsafe_func(repository_dir: str):
    samples = list()
    for dir_path, _, files in os.walk(repository_dir):
        for file_name in files:
            if file_name[-3:] != ".rs":
                continue
            file_path = os.path.join(dir_path, file_name)
            relative_file = os.path.relpath(os.path.join(dir_path, file_name), repository_dir)
            if "test" in file_path or "example" in file_path or "bench" in file_path:
                continue
            tree: Tree = get_syntax_tree(file_path)
            captures = query.captures(tree.root_node)
            for node, _ in captures:
                # tranverse cursor to father function
                fn_node = find_parent_fn_node(node)
                if fn_node is None:
                    continue

                # filter unsafe block in unsafe function
                if not node_of_unsafe_func(fn_node):
                    continue

                # filter private functions
                if not node_of_pub_func(fn_node):
                    continue

                duplicated = False
                for s in samples:
                    if s["relative_file"] == relative_file and s["start_line"] == fn_node.start_point[0]:
                        duplicated = True
                        break
                if duplicated:
                    continue
        
                # filter extern C functions
                if "extern \"C\"" in fn_node.text.decode("utf-8"):
                    continue

                samples.append(
                    {
                        "relative_file": relative_file,
                        "start_line": fn_node.start_point[0], 
                        "start_byte": fn_node.start_byte,
                        "end_line": fn_node.end_point[0],
                        "end_byte": fn_node.end_byte,
                    }
                )
    print(f"Got {len(samples)} unsafe functions in {repository_dir}")
    return samples

## Get top-500 crates
### Clone target repositories

In [4]:
top_crate_file = "data/crate_meta/top_crates.txt"
crates_repo_dir = "data/crates_repo"
risky_func_dir = "data/risky_func"
unsafe_func_dir = "data/unsafe_func"
dir_check(crates_repo_dir)
dir_check(risky_func_dir)
dir_check(unsafe_func_dir)

In [5]:
# filter the crates (top-n)
top_repos = filter_crates(crates_info_file, top_crate_file, (0, 500))

356
[6, 3, 5, 130]


In [ ]:
for repo_path in top_repos:
    repo_dir = os.path.join(crates_repo_dir, ".".join(repo_path.split("/")))
    clone_repo(repo_path, repo_dir)

### Get risky and unsafe samples

In [ ]:
for repo_path in top_repos:
    repo_name = ".".join(repo_path.split("/"))
    repo_dir = os.path.join(crates_repo_dir, repo_name)
    unsafe_func = find_unsafe_func(repo_dir)
    json.dump(unsafe_func, safe_open(os.path.join(unsafe_func_dir, repo_name + ".json"), "w"), indent=2)
    risky_func = find_risky_func(repo_dir)
    json.dump(risky_func, safe_open(os.path.join(risky_func_dir, repo_name + ".json"), "w"), indent=2)

## Scan top 500 - 1000

In [6]:
scan_crate_file = "data/crate_meta/scan_crates.txt"
scan_repo_dir = "data/scan_repo"
scan_func_dir = "data/scan_func"
dir_check(scan_repo_dir)
dir_check(scan_func_dir)

In [7]:
# filter the crates (top-n)
scan_target = filter_crates(crates_info_file, scan_crate_file, (500, 2000))

878
[78, 29, 31, 484]


In [8]:
scan_target = [t for t in scan_target if t not in top_repos]
len(scan_target)

827

In [9]:
for repo_path in scan_target:
    repo_dir = os.path.join(scan_repo_dir, ".".join(repo_path.split("/")))
    clone_repo(repo_path, repo_dir)

wasmerio/wasmer clone failed!
jmap-rs/json-pointer clone failed!
paritytech/pariry-common clone failed!


### Get risky samples

In [ ]:
for repo_path in scan_target:
    repo_name = ".".join(repo_path.split("/"))
    repo_dir = os.path.join(scan_repo_dir, repo_name)
    risky_func = find_risky_func(repo_dir)
    json.dump(risky_func, safe_open(os.path.join(scan_func_dir, repo_name + ".json"), "w"), indent=2)